## Setup

In [2]:
from dotenv import load_dotenv
from utils import chat_interface

c:\dev\miniconda\envs\langchain-agent-legacy\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
load_dotenv()

True

## Run

In [4]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents
# IDEALLY YOUR ONLY IMPORT HERE IS:
from agentic.workflow import orchestrator
import warnings
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

## End-to-End Demo

### Scenario 1: Successful AI resolution (RAG hit)
Classifier -> Resolver (search_knowledge hit) -> Supervisor (approved) -> END

In [ ]:
chat_interface(orchestrator, "demo_1", "f556c0")
# Input: My QR code won't scan at the venue entrance.

User: My QR code won't scan at the venue entrance.
{"timestamp": "2026-09-30T22:26:49.549141", "node": "classifier", "category": "technical", "urgency": "medium", "needs_escalation": false}
{"timestamp": "2026-09-30T22:26:55.100720", "node": "resolver", "attempt": 1, "tools_called": [{"tool": "search_knowledge", "args": {"query": "QR code won't scan venue entrance"}}], "classifier_flagged": false, "resolver_flagged": false, "final_needs_escalation": false}
{"timestamp": "2026-09-30T22:26:56.127514", "node": "supervisor", "decision": "approve"}
Assistant: If your QR code won't scan at the venue entrance, please try refreshing the app or checking your internet connection. Make sure you're viewing the correct reservation under 'My Reservations'. Sometimes lowering the screen brightness or cleaning the screen can help with scanning issues. 

As a backup, you can show your reservation confirmation email along with a photo ID to the venue staff for entry. If you continue to experience proble

### Scenario 2: Escalation due to no relevant knowledge found (RAG miss)
Classifier -> Resolver (search_knowledge miss, needs_escalation=True) -> Supervisor -> Escalation -> interrupt -> END

In [ ]:
chat_interface(orchestrator, "demo_2", "a4ab87")
# Input: Someone next to me was very loud during last week's concert and I want to file a complaint.

User: Someone next to me was very loud during last week's concert and I want to file a complaint.
{"timestamp": "2026-09-30T22:27:13.120190", "node": "classifier", "category": "general", "urgency": "low", "needs_escalation": false}
{"timestamp": "2026-09-30T22:27:18.211662", "node": "resolver", "attempt": 1, "tools_called": [{"tool": "search_knowledge", "args": {"query": "file complaint noisy concert"}}], "classifier_flagged": false, "resolver_flagged": true, "final_needs_escalation": true}
{"timestamp": "2026-09-30T22:27:19.370097", "node": "supervisor", "decision": "reject", "reason": "Response lacks clear guidance on how to formally file a complaint about noise and is too vague."}

[Escalation] Human review required:
{'category': 'general', 'sentiment': 'negative', 'reason_for_escalation': 'Response lacks clear guidance on how to formally file a complaint about noise and is too vague.', 'draft_answer': "I couldn't find specific guidelines on filing a complaint about noise during a c

In [ ]:
# When Escalation Occurred, Human response
# Hi, I'm sorry your experience at the concert was affected by another attendee's noise. To help us look into this properly, could you share:  1. The event name and date 2. Roughly where you were seated or standing 3. A brief description of what happened  I'll pass this along to our venue partnerships team for review. While we don't have a formal noise complaint process listed for this, we do take feedback like this seriously when working with venues on future events. Thank you for taking the time to let us know.


### Scenario 3: Returning customer — repeated issue raises urgency

In [ ]:
chat_interface(orchestrator, "demo_3", "f556c0")
# Input: My QR code failed again today, same as before.

User: My QR code failed again today, same as before.
{"timestamp": "2026-09-30T22:30:08.656805", "node": "classifier", "category": "technical", "urgency": "medium", "needs_escalation": false}
{"timestamp": "2026-09-30T22:30:15.604141", "node": "resolver", "attempt": 1, "tools_called": [{"tool": "search_knowledge", "args": {"query": "QR code failed"}}], "classifier_flagged": false, "resolver_flagged": true, "final_needs_escalation": true}
{"timestamp": "2026-09-30T22:30:16.737931", "node": "supervisor", "decision": "approve"}

[Escalation] Human review required:
{'category': 'technical', 'sentiment': 'negative', 'reason_for_escalation': 'No relevant knowledge base article addressing repeated QR code failure issues specifically.', 'draft_answer': "It seems that the specific issue regarding QR code failures doesn't have a detailed resolution in the knowledge base articles available. However, if your QR code is not working, please make sure to refresh the app and check your internet connec

In [ ]:
# When Escalation Occurred, Human response
# Hi, I'm sorry this keeps happening — I can see this is a repeat of the same QR code issue.  In the meantime, please try: 1. Refreshing the app and checking your internet connection. 2. Confirming you're viewing the correct reservation under 'My Reservations'. 3. Lowering screen brightness or cleaning the screen to reduce glare. 4. If it still fails, show venue staff your confirmation email and a photo ID as a backup for entry.  Since this has happened more than once, I've flagged your account so our technical team can investigate what's causing the repeat failure, rather than just treating it as a one-off. You'll hear back from us once we identify the cause.


### Scenario 4: High urgency ticket routed directly to escalation

In [ ]:
chat_interface(orchestrator, "demo_4", "f556c0")
# Input: My payment was charged twice right now and I need this fixed urgently.

User: My payment was charged twice right now and I need this fixed urgently.
{"timestamp": "2026-09-30T22:34:44.393079", "node": "classifier", "category": "billing", "urgency": "high", "needs_escalation": true}
{"timestamp": "2026-09-30T22:34:48.176431", "node": "resolver", "attempt": 1, "tools_called": [{"tool": "search_knowledge", "args": {"query": "duplicate payment charge"}}], "classifier_flagged": true, "resolver_flagged": true, "final_needs_escalation": true}
{"timestamp": "2026-09-30T22:34:49.518893", "node": "supervisor", "decision": "reject", "reason": "The response does not provide any helpful information or guidance to the customer."}

[Escalation] Human review required:
{'category': 'billing', 'sentiment': 'negative', 'reason_for_escalation': 'The response does not provide any helpful information or guidance to the customer.', 'draft_answer': 'No relevant information was found for this request.'}
{"timestamp": "2026-09-30T22:36:08.793866", "node": "escalation", "resolved_by

In [ ]:
# When Escalation Occurred, Human response
# Hi, I'm sorry to hear about the duplicate charge — I understand this is urgent and want to help resolve it as quickly as possible.  I've flagged this for our billing team to investigate right away. Here's what happens next: 1. We'll verify the duplicate charge within the next few hours. 2. If confirmed, the extra amount will be refunded to your original payment method within 3-5 business days. 3. You'll get a confirmation email once it's processed.  To speed things up, could you confirm the approximate time of the charge and the last 4 digits of the card used?  Thank you for your patience while we sort this out.

In [16]:
# Inspect the full state history of a specific ticket (thread_id) — 
# demonstrates that workflow state (messages, tool usage, decisions) can be
# inspected per-thread for auditing/debugging purposes.
history = list(orchestrator.get_state_history(
    config={"configurable": {"thread_id": "demo_1"}}  # 실제 사용한 thread_id로 교체
))

print(f"Total checkpoints recorded: {len(history)}")
print("\nMost recent state's messages:")
for msg in history[0].values["messages"]:
    print(f"  [{msg.type}] {msg.content[:100]}")

Total checkpoints recorded: 29

Most recent state's messages:
  [human] My QR code won't scan at the venue entrance.
  [ai] If your QR code won't scan at the venue entrance, please try refreshing the app and ensure your inte
  [human] My QR code won't scan at the venue entrance.
  [ai] If your QR code isn't scanning at the venue entrance, try refreshing the CultPass app or checking yo
  [human] My QR code won't scan at the venue entrance.
  [human] My QR code won't scan at the venue entrance.
  [ai] If your QR code won't scan at the venue, try refreshing the app or checking your internet connection
  [human] My QR code won't scan at the venue entrance.
  [ai] If your QR code won't scan at the venue entrance, please try refreshing the app or checking your int
  [ai] If your QR code won't scan at the venue entrance, please try refreshing the app or checking your int
  [human] My QR code won't scan at the venue entrance.
  [ai] If your QR code won't scan at the venue entrance, please try 

In [17]:
# Inspect the full state history of a specific ticket (thread_id) — 
# demonstrates that workflow state (messages, tool usage, decisions) can be
# inspected per-thread for auditing/debugging purposes.
history = list(orchestrator.get_state_history(
    config={"configurable": {"thread_id": "demo_2"}}  # 실제 사용한 thread_id로 교체
))

print(f"Total checkpoints recorded: {len(history)}")
print("\nMost recent state's messages:")
for msg in history[0].values["messages"]:
    print(f"  [{msg.type}] {msg.content[:100]}")

Total checkpoints recorded: 11

Most recent state's messages:
  [human] Someone next to me was very loud during last week's concert and I want to file a complaint.
  [ai] Unfortunately, the knowledge base does not provide a specific procedure for filing a complaint about
  [human] Someone next to me was very loud during last week's concert and I want to file a complaint.
  [ai] Hi, I'm sorry your experience at the concert was affected by another attendee's noise. To help us lo


In [20]:
# Inspect the full state history of a specific ticket (thread_id) — 
# demonstrates that workflow state (messages, tool usage, decisions) can be
# inspected per-thread for auditing/debugging purposes.
history = list(orchestrator.get_state_history(
    config={"configurable": {"thread_id": "demo_3"}}  # 실제 사용한 thread_id로 교체
))

print(f"Total checkpoints recorded: {len(history)}")
print("\nMost recent state's messages:")
for msg in history[0].values["messages"]:
    print(f"  [{msg.type}] {msg.content[:100]}")

Total checkpoints recorded: 12

Most recent state's messages:
  [human] My QR code failed again today, same as before.
  [ai] Hi, I'm sorry you're running into this QR code issue again. Here are a few things to try:  1. Refres
  [human] My QR code failed again today, same as before.
  [ai] It seems that the specific issue regarding QR code failures doesn't have a detailed resolution in th
  [ai] Hi, I'm sorry this keeps happening — I can see this is a repeat of the same QR code issue.  In the m


In [21]:
# Inspect the full state history of a specific ticket (thread_id) — 
# demonstrates that workflow state (messages, tool usage, decisions) can be
# inspected per-thread for auditing/debugging purposes.
history = list(orchestrator.get_state_history(
    config={"configurable": {"thread_id": "demo_4"}}  # 실제 사용한 thread_id로 교체
))

print(f"Total checkpoints recorded: {len(history)}")
print("\nMost recent state's messages:")
for msg in history[0].values["messages"]:
    print(f"  [{msg.type}] {msg.content[:100]}")

Total checkpoints recorded: 12

Most recent state's messages:
  [human] My payment was charged twice right now and I need this fixed urgently.
  [ai] Hi, I'm sorry to hear about the duplicate charge — I understand this is urgent and I want to make su
  [human] My payment was charged twice right now and I need this fixed urgently.
  [ai] Hi, I'm sorry to hear about the duplicate charge — I understand this is urgent and want to help reso
